In [ ]:
import pickle as pkl
import numpy as np

---
---
---

In [ ]:
def layer_fct_log(grid_fft, level):
    
    '''
    The complete function for layering.
    '''

    level -= 1
    
    # Make a cube leveled on the log scale for all the steps (even if some are empty).
    grid_fft   = np.log10(grid_fft)
    grid_fft  -= np.min(grid_fft)
    grid_fft  /= np.max(grid_fft)
    grid_fft  *= level
    grid_fft //= 1

    # Remove any empty levels and make the counting continuous (w.r.t. integers).
    value_map = {value: i for i, value in enumerate(np.unique(grid_fft))}
    grid_fft = np.vectorize(value_map.get)(grid_fft)

    # Remove the values that are the max (equal to the max lvl+1).
    # They should be very few but at least one if we didn't remove any empty level, which is highly unlikely.
    # This doesn't matter since size**3 doesn't perfectly divide by level anyway in most cases.
    for i0, j0, k0 in np.argwhere(grid_fft == level): grid_fft[i0][j0][k0] = level-1
    
    return grid_fft

In [ ]:
def layer_fct_cdf(grid_fft, level):
    
    '''
    The complete function for layering.
    '''

    size = grid_fft.shape[0]
    N = size**3
    
    ranks = np.argsort(np.argsort(grid_fft.flatten()))   # elements' ranks
    
    cdf_leveled = np.floor((ranks / N) * level).astype(np.int32)
    cdf_leveled = np.clip(cdf_leveled, 0, level - 1)  # no overflow
    
    grid_lvld = cdf_leveled.reshape((size, size, size))

    return grid_lvld

---

This function can be used to check that we respected the distribution for the cdf method.

That is, for a size=512 grid suing level=$10^5$, we should get levels containing 512**3 // level = 1342 cells per level, with a remaining 17728 levels getting an extra one.

In [ ]:
if False:
    counts = np.bincount(grid_lvld.ravel(), minlength=100000)
    non_1342_values = np.where(counts != 1342)[0]
    print("Number of levels with more (or less) values: ", len(non_1342_values))
    print("Values that do not occur exactly 1342 times: ", non_1342_values)

---
---
---